# Healthcare Analytics for Doctor Visits

This project studies associations with doctor visits using only the supplied CSV. All 5,190 original records are retained. No values are filled, rescaled, or fabricated.

Age and income remain in their original numerical scale. The visit observation period and the direction of the health score are not defined in the supplied files. Results are descriptive associations, not causal effects or medical advice.

In [1]:
from pathlib import Path
import pandas as pd
import hashlib
source_name = '1776250375-P2-Healthcare Analytics for Doctor Visits (1).csv'
source = next((p for p in [Path.cwd()/source_name, Path.cwd().parent/source_name] if p.exists()), None)
if source is None:
    raise FileNotFoundError('Keep the project folder beside the original CSV.')
df = pd.read_csv(source)
print('Records:', len(df), 'Columns:', len(df.columns))
print('CSV SHA256:', hashlib.sha256(source.read_bytes()).hexdigest())
print(df.head().to_string(index=False))

Records: 5190 Columns: 13
CSV SHA256: 6967773d1f794bc32138478352c3ed507d87faedc73174b1cd179bd11efab225
 Unnamed: 0  visits gender  age  income  illness  reduced  health private freepoor freerepat nchronic lchronic
          1       1 female 0.19    0.55        1        4       1     yes       no        no       no       no
          2       1 female 0.19    0.45        1        2       1     yes       no        no       no       no
          3       1   male 0.19    0.90        3        0       0      no       no        no       no       no
          4       1   male 0.19    0.15        1        0       0      no       no        no       no       no
          5       1   male 0.19    0.45        2        5       1      no       no        no      yes       no


## Data quality and preparation
The first column is an identifier, excluded from numerical comparisons. Matching measurements under different record IDs are retained. Zero income is kept as recorded because its meaning is not explained. No reference specifies an age or income conversion.

In [2]:
print('Missing cells:', int(df.isna().sum().sum()))
print('Duplicate IDs:', int(df.iloc[:, 0].duplicated().sum()))
print('Duplicate complete rows:', int(df.duplicated().sum()))
print('Repeated measurements excluding ID:', int(df.iloc[:, 1:].duplicated().sum()))
print(df[['visits','age','income','illness','reduced','health']].describe().to_string())

Missing cells: 0
Duplicate IDs: 0
Duplicate complete rows: 0
Repeated measurements excluding ID: 1320
            visits          age       income      illness      reduced       health
count  5190.000000  5190.000000  5190.000000  5190.000000  5190.000000  5190.000000
mean      0.301734     0.406385     0.583160     1.431985     0.861850     1.217534
std       0.798134     0.204782     0.368907     1.384152     2.887628     2.124266
min       0.000000     0.190000     0.000000     0.000000     0.000000     0.000000
25%       0.000000     0.220000     0.250000     0.000000     0.000000     0.000000
50%       0.000000     0.320000     0.550000     1.000000     0.000000     0.000000
75%       0.000000     0.620000     0.900000     2.000000     0.000000     2.000000
max       9.000000     0.720000     1.500000     5.000000    14.000000    12.000000


## Overall doctor visits
Average visits = total visits divided by all records. Share with a visit = records with visits greater than zero divided by all records. The CSV provides no time period for these visits.

In [3]:
print('Total visits:', int(df.visits.sum()))
print('Average visits per record:', round(df.visits.mean(), 6))
print('Median visits:', df.visits.median())
print('Records with zero visits:', int((df.visits == 0).sum()))
print('Share with at least one visit (%):', round((df.visits > 0).mean()*100, 6))
print(df.visits.value_counts().sort_index().to_string())

Total visits: 1566
Average visits per record: 0.301734
Median visits: 0.0
Records with zero visits: 4141
Share with at least one visit (%): 20.211946
visits
0    4141
1     782
2     174
3      30
4      24
5       9
6      12
7      12
8       5
9       1


## Doctor visits by gender
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [4]:
summary = df.groupby('gender').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

        records  total_visits  mean_visits  visitors  share_with_visit_pct
gender                                                                    
female     2702           978     0.361954       660             24.426351
male       2488           588     0.236334       389             15.635048


## Doctor visits by illness
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [5]:
summary = df.groupby('illness').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

         records  total_visits  mean_visits  visitors  share_with_visit_pct
illness                                                                    
0           1554           122     0.078507       105              6.756757
1           1638           484     0.295482       321             19.597070
2            946           382     0.403805       258             27.272727
3            542           228     0.420664       166             30.627306
4            274           158     0.576642       100             36.496350
5            236           192     0.813559        99             41.949153


## Doctor visits by reduced
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [6]:
summary = df.groupby('reduced').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

         records  total_visits  mean_visits  visitors  share_with_visit_pct
reduced                                                                    
0           4454           819     0.183880       676             15.177369
1            177            63     0.355932        48             27.118644
2            108            62     0.574074        45             41.666667
3             74            81     1.094595        44             59.459459
4             45            36     0.800000        25             55.555556
5             40            51     1.275000        25             62.500000
6             17            20     1.176471        12             70.588235
7             38            45     1.184211        23             60.526316
8             17            20     1.176471        10             58.823529
9              7            12     1.714286         4             57.142857
10            12            24     2.000000         9             75.000000
11          

## Doctor visits by nchronic
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [7]:
summary = df.groupby('nchronic').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

          records  total_visits  mean_visits  visitors  share_with_visit_pct
nchronic                                                                    
no           3098           843     0.272111       554             17.882505
yes          2092           723     0.345602       495             23.661568


## Doctor visits by lchronic
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [8]:
summary = df.groupby('lchronic').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

          records  total_visits  mean_visits  visitors  share_with_visit_pct
lchronic                                                                    
no           4585          1201     0.261941       837             18.255180
yes           605           365     0.603306       212             35.041322


## Doctor visits by private
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [9]:
summary = df.groupby('private').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

         records  total_visits  mean_visits  visitors  share_with_visit_pct
private                                                                    
no          2892           889     0.307400       589             20.366528
yes         2298           677     0.294604       460             20.017406


## Doctor visits by freepoor
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [10]:
summary = df.groupby('freepoor').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

          records  total_visits  mean_visits  visitors  share_with_visit_pct
freepoor                                                                    
no           4968          1531     0.308172      1029             20.712560
yes           222            35     0.157658        20              9.009009


## Doctor visits by freerepat
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [11]:
summary = df.groupby('freerepat').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

           records  total_visits  mean_visits  visitors  share_with_visit_pct
freerepat                                                                    
no            4099          1057     0.257868       704             17.174921
yes           1091           509     0.466544       345             31.622365


## Doctor visits by age
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [12]:
summary = df.groupby('age').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

      records  total_visits  mean_visits  visitors  share_with_visit_pct
age                                                                     
0.19      752           160     0.212766       114             15.159574
0.22     1213           244     0.201154       174             14.344600
0.27      523           135     0.258126        87             16.634799
0.32      301            73     0.242525        50             16.611296
0.37      146            36     0.246575        26             17.808219
0.42      126            40     0.317460        22             17.460317
0.47      181            49     0.270718        30             16.574586
0.52      222            92     0.414414        54             24.324324
0.57      273           109     0.399267        72             26.373626
0.62      316           112     0.354430        77             24.367089
0.67      315           119     0.377778        69             21.904762
0.72      822           397     0.482968       274 

## Doctor visits by income
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [13]:
summary = df.groupby('income').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

        records  total_visits  mean_visits  visitors  share_with_visit_pct
income                                                                    
0.00         79            29     0.367089        13             16.455696
0.01         35            17     0.485714         5             14.285714
0.06         80            16     0.200000        12             15.000000
0.15        249            87     0.349398        54             21.686747
0.25       1195           503     0.420921       336             28.117155
0.35        462           166     0.359307       101             21.861472
0.45        400           122     0.305000        86             21.500000
0.55        467           118     0.252677        80             17.130621
0.65        455           112     0.246154        85             18.681319
0.75        441           108     0.244898        75             17.006803
0.90        589           122     0.207131        86             14.601019
1.10        361          

## Doctor visits by health
Group averages use each group's own record count. Group differences are unadjusted. Different healthcare support or chronic flags can overlap and must not be added together.

In [14]:
summary = df.groupby('health').agg(records=('visits','size'), total_visits=('visits','sum'), mean_visits=('visits','mean'), visitors=('visits',lambda s: int((s>0).sum())))
summary['share_with_visit_pct'] = 100*summary.visitors/summary.records
assert summary.records.sum() == len(df)
assert summary.total_visits.sum() == df.visits.sum()
print(summary.to_string(float_format=lambda v: f'{v:.6f}'))

        records  total_visits  mean_visits  visitors  share_with_visit_pct
health                                                                    
0          3026           603     0.199273       455             15.036352
1           823           276     0.335358       193             23.450790
2           446           170     0.381166       109             24.439462
3           273           114     0.417582        70             25.641026
4           187           100     0.534759        52             27.807487
5           132            70     0.530303        37             28.030303
6           104            65     0.625000        41             39.423077
7            61            56     0.918033        30             49.180328
8            42            20     0.476190        13             30.952381
9            32            21     0.656250        12             37.500000
10           21            32     1.523810        12             57.142857
11           24          

## Activity affected by illness
This explicit grouping compares reduced = 0 with reduced > 0. It changes only the grouping, not the source values.

In [15]:
affected = df.assign(activity_affected=df.reduced.gt(0).map({False:'No reduced days', True:'One or more reduced days'}))
print(affected.groupby('activity_affected').agg(records=('visits','size'),total_visits=('visits','sum'),mean_visits=('visits','mean')).to_string())

                          records  total_visits  mean_visits
activity_affected                                           
No reduced days              4454           819     0.183880
One or more reduced days      736           747     1.014946


## Conclusion
Records with more illnesses, reduced activity days, or lchronic = yes generally show higher average visits in this sample. Females average more visits than males. Healthcare support comparisons vary by flag and do not establish access, need, quality of care, or program effectiveness. Age, income and health are reported on their original scales. No population weights, observation dates, causal adjustment, or predictive model are supplied or assumed.